# Text-to-text LLM Evaluation

## Import packages

In [275]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import json
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass
from pathlib import Path

from langchain_ollama.llms import OllamaLLM as Ollama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from neurorag.models.OpenRouter import OpenRouter
import transformers
import torch

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  summac_zs_metric,
  summac_conv_metric,
)

## Disable warnings

In [276]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

**Note:** FActScore metric requires an OpenAI API key to function properly, as it uses GPT models for fact extraction and verification.

## Import packages

In [277]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Define evaluation function

In [278]:
def eval_rag(chain) -> float:
  dataset_df = pd.read_csv('../datasets/mediqa.csv')
  expected_answers = dataset_df['answer']
  predicted_answers = []

  for index, row in tqdm(list(dataset_df.iterrows()), desc='Questions'):
    question = row['question']
    llm_answer = chain.invoke({'query': question})
    predicted_answers.append(llm_answer)

  cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
  factscore = factscore_metric(expected_answers, predicted_answers)

  return cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore

## Define prompt

In [279]:
template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
Keep the answer verbose, with a minimum of three paragraphs.

QUERY: {query}

First, identify the key scientific concepts and data points in the CONTEXT that relate to the QUERY.
Then, analyze how these concepts connect to form a comprehensive answer.
Finally, synthesize your findings into a detailed response.
"""

prompt = PromptTemplate(
  template=template,
  input_variables=['query'],
)

## Setup LLMs

### Llama 3.3 70B Instruct

In [280]:
def get_llama3_3_70b_instruct_llm(temperature=0.0):
  return OpenRouter(model='meta-llama/llama-3.3-70b-instruct', temperature=temperature)

### Mistral Large

In [281]:
def get_mistral_large_llm(temperature=0.0):
  return OpenRouter(
    model='mistralai/mistral-large',
    temperature=temperature,
  )

### GPT-4.1

In [282]:
def get_chatgpt_4_1_llm(temperature=0.0):
  return OpenRouter(
    model='openai/gpt-4.1',
    temperature=temperature,
  )

### OpenBioLLM 70B Q2_k

In [283]:
def get_openbiollm_70_Q2k_llm(temperature=0.0):
  return Ollama(model='taozhiyuai/openbiollm-llama-3:70b_q2_k', temperature=temperature)

### Biomistral 7B Q4_k_m

In [284]:
def get_biomistral_q4_k_m_llm(temperature=0.0):
  return Ollama(model='cniongolo/biomistral', temperature=temperature)

### OpenBioLLM Llama3 70B (HuggingFace)

In [285]:
def get_openbiollm_llama3_70b_llm(temperature=0.0):
  model_id = "aaditya/OpenBioLLM-Llama3-70B"
  
  pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
  )
  
  # Configure the pipeline with terminators and generation parameters
  terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
  ]
  
  # Create HuggingFace pipeline wrapper for LangChain
  hf_pipeline = HuggingFacePipeline(
    pipeline=pipeline,
    pipeline_kwargs={
      "max_new_tokens": 512,
      "eos_token_id": terminators,
      "do_sample": True if temperature > 0 else False,
      "temperature": temperature if temperature > 0 else None,
      "top_p": 0.9,
    }
  )
  
  return hf_pipeline

## Evaluate the models

### Load QA dataset

In [286]:
mediqa_df = pd.read_csv('../datasets/neurobiology_mediqa.csv')
mediqa_df

,question,answer
0,SSPE. My son is 33years of age and did not hav...,Subacute sclerosing panencephalitis: Subacute ...
1,Homozygout MTHFR A1298C Health Issues and long...,MTHFR gene variant (Inheritance): Because each...
2,What is Stroke?,Stroke: A stroke occurs when the blood supply ...
3,What causes Stroke?,Ischemic Stroke (Summary): Summary A stroke is...
4,What are the symptoms of Stroke?,What are the symptoms of Stroke?: The signs an...
5,What are the treatments of Stroke?,Stroke (Treatment): A stroke is a medical emer...
6,What is Dementia?,Dementia (WHAT IS DEMENTIA?): Dementia is the ...
7,What causes Dementia?,What causes Dementia?: Dementia usually occurs...
8,What are the symptoms of Dementia?,Dementia (Symptoms): Dementia symptoms include...
9,How to diagnose Dementia?,Dementia (Diagnosis): Diagnosing dementia and ...


### Setup experiment grid search parameters

In [287]:
llms = (
  # ('Llama 3.3 70B Instruct', get_llama3_3_70b_instruct_llm()),
  ('GPT-4.1', get_chatgpt_4_1_llm()),
  # ('Mistral Large', get_mistral_large_llm()),
  # ('OpenBioLLM Llama3 70B', get_openbiollm_llama3_70b_llm()),
  # ('Biomistral 7B Q4_k_m', get_biomistral_q4_k_m_llm()),
)

### Load cached RAGs responses

In [288]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-llm-evaluation'

del cache[CACHE_KEY]

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache.keys())

3

### Conduct the grid search

In [289]:
df = pd.DataFrame()

questions = mediqa_df['question'].tolist()
expected_answers = mediqa_df['answer'].tolist()

for llm_name, llm in llms:
  chain = prompt | llm | StrOutputParser()

  predicted_answers = []

  if llm_name not in cache[CACHE_KEY]:
    cache[CACHE_KEY][llm_name] = {}

  for question in tqdm(questions, desc='Questions'):
    if question not in cache[CACHE_KEY][llm_name]:
      cache[CACHE_KEY][llm_name][question] = chain.invoke(question)

    predicted_answers.append(cache[CACHE_KEY][llm_name][question])

    with open(cache_path, 'w') as f:
      json.dump(cache, f)

  # Evaluate metrics
  cos_sim = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
  factscore = factscore_metric(expected_answers, predicted_answers)

  # Save results
  row = pd.DataFrame({
    'llm': llm_name,
    'cos_sim': cos_sim,
    'bleu': bleu_score,
    'rogue_1': rogue_1_score,
    'rogue_l': rogue_l_score,
    'factscore': factscore,
  }, index=[0])
  df = pd.concat([df, row], ignore_index=True)

df.sort_values(by='cos_sim', ascending=False)

Questions:   0%|          | 0/19 [00:00<?, ?it/s]

2026-01-15 18:53:34,577 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
Questions: 100%|██████████| 19/19 [05:36<00:00, 17.73s/it]
2026-01-15 18:59:11,083 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-01-15 18:59:11,573 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-01-15 18:59:11,939 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-01-15 18:59:12,355 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-01-15 18:59:12,905 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-01-15 18:59:13,485 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-01-15 18:59:13,989 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-01-15 18:59:14,511 - INFO - HTTP Request: POST htt

Extracting facts from generations...


0it [00:00, ?it/s]


Generating decisions...


...

0it [00:00, ?it/s]


Generating decisions...


0it [00:00, ?it/s]

Error computing FActScore for pair: division by zero


,llm,cos_sim,bleu,rogue_1,rogue_l,factscore
0,GPT-4.1,0.77521,0.027391,0.305693,0.157073,0.0


In [290]:
print(cache['text-to-text-neurorag-evaluation']['What is Stroke?'])

A stroke is a serious medical emergency that occurs when the blood supply to a part of the brain is interrupted or significantly reduced, depriving brain tissue of essential oxygen and nutrients. This disruption leads to the rapid death of brain cells and can result in permanent neurological damage or death if not treated promptly. Strokes are broadly classified into two main types: ischemic and hemorrhagic. 

Ischemic strokes, which account for about 80% of all cases, are caused by blockages in the blood vessels supplying the brain, most commonly due to blood clots or atherosclerotic plaque buildup. These can be further divided into thrombotic strokes (caused by local blockages in cerebral arteries) and embolic strokes (caused by clots or debris traveling from elsewhere in the body, often the heart, to the brain). Hemorrhagic strokes, making up roughly 20% of cases, occur when a blood vessel in the brain ruptures, leading to bleeding within or around the brain and increased intracrani

In [291]:
print(cache['text-to-text-llm-evaluation']['GPT-4.1']['What is Stroke?'])

Certainly! Let’s break down the answer as requested:

Key Scientific Concepts and Data Points from the Context:
- Stroke is a medical emergency that occurs when the blood supply to part of the brain is interrupted or reduced.
- There are two main types of stroke: ischemic (caused by a blockage, such as a blood clot, in an artery supplying the brain) and hemorrhagic (caused by a blood vessel in the brain rupturing and bleeding).
- Without adequate blood flow, brain cells begin to die within minutes due to lack of oxygen and nutrients.
- Symptoms of stroke can include sudden numbness or weakness (especially on one side of the body), confusion, trouble speaking or understanding speech, vision problems, dizziness, and loss of balance or coordination.
- Stroke is a leading cause of disability and death worldwide.
- Risk factors for stroke include high blood pressure, smoking, diabetes, high cholesterol, heart disease, and certain lifestyle factors.

Analysis of How These Concepts Connect:
T

In [292]:
print(expected_answers[2])

Stroke: A stroke occurs when the blood supply to part of the brain is suddenly interrupted or when a blood vessel in the brain bursts, spilling blood into the spaces surrounding brain cells. Brain cells die when they no longer receive oxygen and nutrients from the blood or there is sudden bleeding into or around the brain. The symptoms of a stroke include sudden numbness or weakness, especially on one side of the body; sudden confusion or trouble speaking or understanding speech; sudden trouble seeing in one or both eyes; sudden trouble with walking, dizziness, or loss of balance or coordination; or sudden severe headache with no known cause. There are two forms of stroke: ischemic - blockage of a blood vessel supplying the brain, and hemorrhagic - bleeding into or around the brain. Generally there are three treatment stages for stroke: prevention, therapy immediately after the stroke, and post-stroke rehabilitation. Therapies to prevent a first or recurrent stroke are based on treatin